In [ ]:
from google.colab import files
files.upload()

Saving BCICIV_2b_gdf.zip to BCICIV_2b_gdf.zip


In [ ]:
!unzip BCICIV_2b_gdf.zip -d /content/data

Archive:  BCICIV_2b_gdf.zip
  inflating: /content/data/B0101T.gdf  
  inflating: /content/data/B0102T.gdf  
  inflating: /content/data/B0103T.gdf  
  inflating: /content/data/B0104E.gdf  
  inflating: /content/data/B0105E.gdf  
  inflating: /content/data/B0201T.gdf  
  inflating: /content/data/B0202T.gdf  
  inflating: /content/data/B0203T.gdf  
  inflating: /content/data/B0204E.gdf  
  inflating: /content/data/B0205E.gdf  
  inflating: /content/data/B0301T.gdf  
  inflating: /content/data/B0302T.gdf  
  inflating: /content/data/B0303T.gdf  
  inflating: /content/data/B0304E.gdf  
  inflating: /content/data/B0305E.gdf  
  inflating: /content/data/B0401T.gdf  
  inflating: /content/data/B0402T.gdf  
  inflating: /content/data/B0403T.gdf  
  inflating: /content/data/B0404E.gdf  
  inflating: /content/data/B0405E.gdf  
  inflating: /content/data/B0501T.gdf  
  inflating: /content/data/B0502T.gdf  
  inflating: /content/data/B0503T.gdf  
  inflating: /content/data/B0504E.gdf  
  inflating:

In [ ]:
pip install mne

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 50.9 MB/s eta 0:00:00


In [ ]:
import mne

test_file = "/content/data/B0101T.gdf"  # use any one of your files

raw = mne.io.read_raw_gdf(test_file, preload=False, verbose=False)

events, event_dict = mne.events_from_annotations(raw)

print("Event dictionary:")
print(event_dict)

Used Annotations descriptions: [np.str_('1023'), np.str_('1077'), np.str_('1078'), np.str_('1079'), np.str_('1081'), np.str_('276'), np.str_('277'), np.str_('32766'), np.str_('768'), np.str_('769'), np.str_('770')]
Event dictionary:
{np.str_('1023'): 1, np.str_('1077'): 2, np.str_('1078'): 3, np.str_('1079'): 4, np.str_('1081'): 5, np.str_('276'): 6, np.str_('277'): 7, np.str_('32766'): 8, np.str_('768'): 9, np.str_('769'): 10, np.str_('770'): 11}


/tmp/ipython-input-2160207769.py:5: RuntimeWarning: Highpass cutoff frequency 100.0 is greater than lowpass cutoff frequency 0.5, setting values to 0 and Nyquist.
  raw = mne.io.read_raw_gdf(test_file, preload=False, verbose=False)


In [ ]:
import os
import glob
import numpy as np
import mne

from mne.decoding import CSP
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score

# ================================
# 1. DATA PATH
# ================================
DATA_PATH = "/content/data"   # 🔴 CHANGE if needed

# ================================
# 2. LOAD ONLY TRAINING FILES (T)
# ================================
gdf_files = sorted(glob.glob(os.path.join(DATA_PATH, "B0*T.gdf")))

print("Found training files:", len(gdf_files))
for f in gdf_files[:5]:
    print(os.path.basename(f))

# ================================
# 3. EVENT IDs (BCI IV 2a)
# ================================
EVENT_ID = {
    '769': 0,   # left hand
    '770': 1    # right hand
}

# ================================
# 4. FILTER BANK
# ================================
bands = [
    (4, 8),
    (8, 12),
    (12, 16),
    (16, 20),
    (20, 24),
    (24, 30)
]

X_all = []
y_all = []

# ================================
# 5. PROCESS EACH FILE
# ================================
for file in gdf_files:

    print(f"\nProcessing {os.path.basename(file)}")

    raw = mne.io.read_raw_gdf(file, preload=True, verbose=False)

    # ✅ IMPORTANT: apply filter properly
    raw.filter(4., 38., fir_design='firwin', verbose=False)

    events, event_dict = mne.events_from_annotations(raw, verbose=False)

    # 🔴 Check available events
    available_events = set(event_dict.keys())
    valid_keys = [k for k in EVENT_ID.keys() if k in available_events]

    if len(valid_keys) < 2:
        print("Skipping file (missing classes)")
        continue

    # ================================
    # Epoch extraction
    # ================================
    picks = mne.pick_types(raw.info, eeg=True, exclude='bads')

    epochs = mne.Epochs(
        raw,
        events,
        event_id={k: event_dict[k] for k in valid_keys},
        tmin=0.5,
        tmax=2.5,
        picks=picks,
        baseline=None,
        preload=True,
        verbose=False
    )

    X = epochs.get_data()
    y = epochs.events[:, -1]

    # map labels to 0/1
    label_map = {event_dict[k]: EVENT_ID[k] for k in valid_keys}
    y = np.array([label_map[val] for val in y])

    X_all.append(X)
    y_all.append(y)

# ================================
# 6. CONCATENATE DATA
# ================================
X_all = np.concatenate(X_all, axis=0)
y_all = np.concatenate(y_all, axis=0)

print("\nFinal dataset shape:", X_all.shape)

# ================================
# 7. BUILD FBCSP FEATURES PROPERLY
# ================================
from sklearn.base import BaseEstimator, TransformerMixin

class FBCSP(BaseEstimator, TransformerMixin):
    def __init__(self, bands, n_components=4, sfreq=250):
        self.bands = bands
        self.n_components = n_components
        self.sfreq = sfreq
        self.csp_list = []

    def fit(self, X, y):
        self.csp_list = []
        self.filters_ = []

        for l_freq, h_freq in self.bands:
            X_band = mne.filter.filter_data(
                X,
                sfreq=self.sfreq,
                l_freq=l_freq,
                h_freq=h_freq,
                verbose=False
            )

            csp = CSP(n_components=self.n_components, log=True, norm_trace=False)
            csp.fit(X_band, y)

            self.csp_list.append(csp)

        return self

    def transform(self, X):
        features = []

        for (l_freq, h_freq), csp in zip(self.bands, self.csp_list):
            X_band = mne.filter.filter_data(
                X,
                sfreq=self.sfreq,
                l_freq=l_freq,
                h_freq=h_freq,
                verbose=False
            )

            feat = csp.transform(X_band)
            features.append(feat)

        return np.concatenate(features, axis=1)

# ================================
# 8. PIPELINE (NO LEAKAGE)
# ================================
pipeline = Pipeline([
    ('fbcsp', FBCSP(bands=bands, n_components=4, sfreq=250)),
    ('lda', LinearDiscriminantAnalysis())
])

scores = cross_val_score(pipeline, X_all, y_all, cv=5)

print("\n✅ Proper FBCSP + LDA Accuracy: %.2f%%" % (np.mean(scores)*100))

Found training files: 27
B0101T.gdf
B0102T.gdf
B0103T.gdf
B0201T.gdf
B0202T.gdf

Processing B0101T.gdf


/tmp/ipython-input-2401568837.py:55: RuntimeWarning: Highpass cutoff frequency 100.0 is greater than lowpass cutoff frequency 0.5, setting values to 0 and Nyquist.
  raw = mne.io.read_raw_gdf(file, preload=True, verbose=False)



Processing B0102T.gdf


/tmp/ipython-input-2401568837.py:55: RuntimeWarning: Highpass cutoff frequency 100.0 is greater than lowpass cutoff frequency 0.5, setting values to 0 and Nyquist.
  raw = mne.io.read_raw_gdf(file, preload=True, verbose=False)



Processing B0103T.gdf


...


Processing B0902T.gdf


/tmp/ipython-input-2401568837.py:55: RuntimeWarning: Highpass cutoff frequency 100.0 is greater than lowpass cutoff frequency 0.5, setting values to 0 and Nyquist.
  raw = mne.io.read_raw_gdf(file, preload=True, verbose=False)



Processing B0903T.gdf


/tmp/ipython-input-2401568837.py:55: RuntimeWarning: Highpass cutoff frequency 100.0 is greater than lowpass cutoff frequency 0.5, setting values to 0 and Nyquist.
  raw = mne.io.read_raw_gdf(file, preload=True, verbose=False)



Final dataset shape: (3680, 6, 501)
Computing rank from data with rank=None
    Using tolerance 1.1e-05 (2.2e-16 eps * 6 dim * 8.1e+09  max singular value)
    Estimated rank (data): 6
    data: rank 6 computed from 6 data channels with 0 projectors
Reducing data rank from 6 -> 6
Estimating class=0 covariance using EMPIRICAL
Done.
Estimating class=1 covariance using EMPIRICAL
Done.
Computing rank from data with rank=None
    Using tolerance 8.6e-06 (2.2e-16 eps * 6 dim * 6.4e+09  max singular value)
    Estimated rank (data): 6
    data: rank 6 computed from 6 data channels with 0 projectors
Reducing data rank from 6 -> 6
Estimating class=0 covariance using EMPIRICAL
Done.
Estimating class=1 covariance using EMPIRICAL
Done.
Computing rank from data with rank=None
    Using tolerance 6.8e-06 (2.2e-16 eps * 6 dim * 5.1e+09  max singular value)
    Estimated rank (data): 6
    data: rank 6 computed from 6 data channels with 0 projectors
Reducing data rank from 6 -> 6
Estimating class=0 c

In [ ]:
# ============================================================
# SUBJECT-WISE FBCSP + LDA (LEFT vs RIGHT)
# ============================================================

import os
import glob
import numpy as np
import mne

from mne.decoding import CSP
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.base import BaseEstimator, TransformerMixin

# ================================
# USER SETTINGS
# ================================
DATA_PATH = "/content/data"
SUBJECT = "01"   # 🔴 change subject here

EVENT_ID = {
    '769': 0,   # left
    '770': 1    # right
}

bands = [
    (8, 12),
    (12, 16),
    (16, 20),
    (20, 24),
    (24, 28),
    (28, 32)
]

SFREQ = 250
N_CSP = 6

# ================================
# FIND SUBJECT FILES (T only)
# ================================
pattern = os.path.join(DATA_PATH, f"B{SUBJECT}*T.gdf")
gdf_files = sorted(glob.glob(pattern))

print(f"Subject {SUBJECT} files:", len(gdf_files))

# ================================
# LOAD DATA
# ================================
X_all = []
y_all = []

for file in gdf_files:

    print("Loading:", os.path.basename(file))

    raw = mne.io.read_raw_gdf(file, preload=True, verbose=False)

    # broadband filter
    raw.filter(4., 38., fir_design='firwin', verbose=False)

    events, event_dict = mne.events_from_annotations(raw, verbose=False)

    available = set(event_dict.keys())
    valid_keys = [k for k in EVENT_ID if k in available]

    if len(valid_keys) < 2:
        print("Skipping file (missing class)")
        continue

    picks = mne.pick_types(raw.info, eeg=True, exclude='bads')

    epochs = mne.Epochs(
        raw,
        events,
        event_id={k: event_dict[k] for k in valid_keys},
        tmin=0.5,
        tmax=2.5,
        picks=picks,
        baseline=None,
        preload=True,
        verbose=False
    )

    X = epochs.get_data()
    y = epochs.events[:, -1]

    label_map = {event_dict[k]: EVENT_ID[k] for k in valid_keys}
    y = np.array([label_map[v] for v in y])

    X_all.append(X)
    y_all.append(y)

# concatenate
X_all = np.concatenate(X_all, axis=0)
y_all = np.concatenate(y_all, axis=0)

print("Final shape:", X_all.shape)

# ============================================================
# FBCSP CLASS
# ============================================================

class FBCSP(BaseEstimator, TransformerMixin):
    def __init__(self, bands, n_components=6, sfreq=250):
        self.bands = bands
        self.n_components = n_components
        self.sfreq = sfreq

    def fit(self, X, y):
        self.csp_list_ = []

        for l_freq, h_freq in self.bands:
            X_band = mne.filter.filter_data(
                X,
                sfreq=self.sfreq,
                l_freq=l_freq,
                h_freq=h_freq,
                verbose=False
            )

            csp = CSP(
                n_components=self.n_components,
                log=True,
                norm_trace=False
            )
            csp.fit(X_band, y)
            self.csp_list_.append(csp)

        return self

    def transform(self, X):
        feats = []

        for (l_freq, h_freq), csp in zip(self.bands, self.csp_list_):
            X_band = mne.filter.filter_data(
                X,
                sfreq=self.sfreq,
                l_freq=l_freq,
                h_freq=h_freq,
                verbose=False
            )

            feats.append(csp.transform(X_band))

        return np.concatenate(feats, axis=1)

# ============================================================
# PIPELINE
# ============================================================

pipeline = Pipeline([
    ('fbcsp', FBCSP(bands=bands, n_components=N_CSP, sfreq=SFREQ)),
    ('lda', LinearDiscriminantAnalysis(
        solver='lsqr',
        shrinkage='auto'
    ))
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(pipeline, X_all, y_all, cv=cv)

print("\n✅ Subject-wise Accuracy: %.2f%% ± %.2f%%"
      % (np.mean(scores)*100, np.std(scores)*100))

Subject 01 files: 3
Loading: B0101T.gdf


/tmp/ipython-input-2783185774.py:57: RuntimeWarning: Highpass cutoff frequency 100.0 is greater than lowpass cutoff frequency 0.5, setting values to 0 and Nyquist.
  raw = mne.io.read_raw_gdf(file, preload=True, verbose=False)


Loading: B0102T.gdf


/tmp/ipython-input-2783185774.py:57: RuntimeWarning: Highpass cutoff frequency 100.0 is greater than lowpass cutoff frequency 0.5, setting values to 0 and Nyquist.
  raw = mne.io.read_raw_gdf(file, preload=True, verbose=False)


Loading: B0103T.gdf


/tmp/ipython-input-2783185774.py:57: RuntimeWarning: Highpass cutoff frequency 100.0 is greater than lowpass cutoff frequency 0.5, setting values to 0 and Nyquist.
  raw = mne.io.read_raw_gdf(file, preload=True, verbose=False)


Final shape: (400, 6, 501)
Computing rank from data with rank=None
    Using tolerance 2.3e-06 (2.2e-16 eps * 6 dim * 1.7e+09  max singular value)
    Estimated rank (data): 6
    data: rank 6 computed from 6 data channels with 0 projectors
Reducing data rank from 6 -> 6
Estimating class=0 covariance using EMPIRICAL
Done.
Estimating class=1 covariance using EMPIRICAL
Done.
Computing rank from data with rank=None
    Using tolerance 2.3e-06 (2.2e-16 eps * 6 dim * 1.7e+09  max singular value)
    Estimated rank (data): 6
    data: rank 6 computed from 6 data channels with 0 projectors
Reducing data rank from 6 -> 6
Estimating class=0 covariance using EMPIRICAL
Done.
Estimating class=1 covariance using EMPIRICAL
Done.
Computing rank from data with rank=None
    Using tolerance 2.6e-06 (2.2e-16 eps * 6 dim * 1.9e+09  max singular value)
    Estimated rank (data): 6
    data: rank 6 computed from 6 data channels with 0 projectors
Reducing data rank from 6 -> 6
Estimating class=0 covariance 